## Transform Races Data
- Read bronze races table
- Keep only the columns required for analytics (drop url column)
- Standardise column names using snal case(circuitId -> circuit_id,racename->race_name)
- Rename columns to make them more meaningful(date->race_date)
- Remove duplicate records
- Transform values of columns race_name to Title Case
- Write the transformed data to silver races table

In [0]:
%run ../00-common/01.environment_config

In [0]:
bronze_table = f'{catalog_name}.{bronze_schema}.races'
silver_table = f'{catalog_name}.{silver_schema}.races'

###Step 1 - Read bronze circuits table

In [0]:
#races_df = spark.read.option.('versionAsOf',0).table(bronze_table)

In [0]:
races_df = spark.table(bronze_table)
display(races_df)

### Step 2 - Keep only the columns required for analytics (drop url column)

In [0]:
# circuits_selected_df = circuits_df.select(
#     'circuitId',
#     'circuitName',
#     'lat',
#     'long',
#     'locality',
#     'country',
#     'ingestion_timestamp',
#     'source_file'
# )

In [0]:
from pyspark.sql import functions as F
races_selected_df = races_df.select(
    F.col('season'),
    F.col('round'),
    F.col('raceName'),
    F.col('date'),
    F.col('circuitId'),
    F.col('ingestion_timestamp'),
    F.col('source_file')
)

### Step 3
- Standardise column names using snal case(circuitId -> circuit_id)
- Rename columns to make them more meaningful(lat -> lattitude)

In [0]:
races_renamed_df = races_selected_df \
    .withColumnsRenamed(
        {
        'circuitId':'circuit_id',
        'raceName':'race_name',
        'date':'race_date'
        }
        )



### Step 6 - Remove duplicate records

In [0]:
#TABLE HAS COMPOSITE PRIMARY KEY
races_distinct_df = races_renamed_df.dropDuplicates(['season','round'])

#display(races_distinct_df)


### Step 7 -Transform values of columns race_name to Title Case

In [0]:
races_final_df = (
    races_distinct_df
    .withColumn("race_name",F.initcap(F.col("race_name")))
    )
#display(races_final_df)

### Step 8 - Write the transformed data to silver circuits table


In [0]:
(
    races_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(silver_table)

)

In [0]:
display(spark.table(silver_table))